In [1]:
import torch
import torchvision
print("Torch OK:", torch.__version__)
print("Torchvision OK:", torchvision.__version__)


Torch OK: 2.9.1+cpu
Torchvision OK: 0.24.1+cpu


In [2]:
import torch
import torch.nn as nn
import torchvision.models as models


In [5]:
class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.fc = nn.Sequential(
            nn.Conv2d(in_planes, in_planes // ratio, 1, bias=False),
            nn.ReLU(),
            nn.Conv2d(in_planes // ratio, in_planes, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        return self.sigmoid(avg_out + max_out)


In [7]:
class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x = torch.cat([avg_out, max_out], dim=1)
        return self.sigmoid(self.conv(x))


In [8]:
class CBAM(nn.Module):
    def __init__(self, in_planes):
        super(CBAM, self).__init__()
        self.ca = ChannelAttention(in_planes)
        self.sa = SpatialAttention()

    def forward(self, x):
        x = self.ca(x) * x
        x = self.sa(x) * x
        return x


In [9]:
class DR_EfficientNet_CBAM(nn.Module):
    def __init__(self, num_classes=5):
        super(DR_EfficientNet_CBAM, self).__init__()

        self.efficientnet = models.efficientnet_b0(pretrained=True)

        self.features = self.efficientnet.features  # 1280 channels
        self.cbam = CBAM(1280)
        self.pool = nn.AdaptiveAvgPool2d(1)

        self.classifier = nn.Linear(1280, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = self.cbam(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

efficient_model = DR_EfficientNet_CBAM(num_classes=5).to(device)

dummy = torch.randn(1, 3, 224, 224).to(device)
out = efficient_model(dummy)

print("EfficientNet output shape:", out.shape)


EfficientNet output shape: torch.Size([1, 5])


In [1]:
import os, sys

PROJECT_ROOT = os.path.abspath(".")
sys.path.insert(0, PROJECT_ROOT)

print("Working directory:", PROJECT_ROOT)
print("Files here:", os.listdir(PROJECT_ROOT))


Working directory: c:\Users\dhira_5fqr2uc\Desktop\Diabetic _Retinography_Project
Files here: ['app.py', 'clean_dataset.py', 'Datasets', 'data_loader.py', 'dr_env', 'DR_Preprocessing.ipynb', 'efficientnet_cbam_best.pth', 'evaluate.ipynb', 'grad cam.ipynb', 'gradcam.py', 'model_notebook.ipynb', 'preprocess_dataset.py', 'training.ipynb', 'training.py', 'training_utils.py', '__pycache__']
